In [27]:
import pandas as pd

tabular = pd.read_csv("data/train/train_tabular.csv")
notes = pd.read_csv("data/train/train_notes.csv")

merged = pd.merge(
    tabular,
    notes,
    how="inner",
    on="flight_id",
)

merged["maintenance_note"] = merged["maintenance_note"].fillna("").astype(str)
merged["failure_mode"] = merged["failure_mode"].fillna("none").astype(str)


In [28]:
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.base import BaseEstimator, TransformerMixin
import numpy as np


class MaintenanceNoteTargetEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, smoothing=0.0):
        self.smoothing = smoothing

    def fit(self, X, y=None):
        X = X.copy()
        if isinstance(X, pd.DataFrame):
            note_series = X.iloc[:, 0]
            failure_series = X.iloc[:, 1] if X.shape[1] > 1 else pd.Series(["none"] * len(X), index=X.index)
        else:
            note_series = pd.Series(X[:, 0])
            failure_series = pd.Series(X[:, 1]) if X.shape[1] > 1 else pd.Series(["none"] * len(X))

        y = np.asarray(y)
        self.global_mean_ = float(np.nanmean(y)) if y.size else 0.5
        self.global_battery_rate_ = float(np.mean(np.array(failure_series == "battery", dtype=float))) if len(failure_series) else 0.5
        self.global_bearing_rate_ = float(np.mean(np.array(failure_series == "bearing", dtype=float))) if len(failure_series) else 0.5

        self.note_means_ = (
            pd.Series(y, index=note_series)
            .groupby(level=0)
            .mean()
            .to_dict()
        )
        self.note_counts_ = (
            pd.Series(y, index=note_series)
            .groupby(level=0)
            .size()
            .to_dict()
        )
        self.note_battery_rates_ = (
            pd.Series((failure_series == "battery").astype(float), index=note_series)
            .groupby(level=0)
            .mean()
            .to_dict()
        )
        self.note_bearing_rates_ = (
            pd.Series((failure_series == "bearing").astype(float), index=note_series)
            .groupby(level=0)
            .mean()
            .to_dict()
        )
        return self

    def transform(self, X):
        X = X.copy()
        if isinstance(X, pd.DataFrame):
            note_series = X.iloc[:, 0]
            failure_series = X.iloc[:, 1] if X.shape[1] > 1 else pd.Series(["none"] * len(X), index=X.index)
        else:
            note_series = pd.Series(X[:, 0])
            failure_series = pd.Series(X[:, 1]) if X.shape[1] > 1 else pd.Series(["none"] * len(X))

        encoded = []
        for note_value, failure_value in zip(note_series, failure_series):
            note_key = "" if pd.isna(note_value) else str(note_value)
            failure_key = "" if pd.isna(failure_value) else str(failure_value)

            count = self.note_counts_.get(note_key, 0)
            mean = self.note_means_.get(note_key, self.global_mean_)
            if self.smoothing > 0 and count > 0:
                target_rate = (count * mean + self.smoothing * self.global_mean_) / (count + self.smoothing)
            else:
                target_rate = mean

            battery_rate = self.note_battery_rates_.get(note_key, self.global_battery_rate_)
            bearing_rate = self.note_bearing_rates_.get(note_key, self.global_bearing_rate_)
            encoded.append([target_rate, battery_rate, bearing_rate])

        return np.array(encoded, dtype=float)

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            input_features = ["maintenance_note", "failure_mode"]
        input_features = np.atleast_1d(input_features)
        return np.array(
            [
                "maintenance_note_failure_rate",
                "maintenance_note_battery_rate",
                "maintenance_note_bearing_rate",
            ],
            dtype=object,
        )


In [29]:
X = merged.drop(columns=["flight_id", "drone_id", "failure_within_horizon"])
y = merged["failure_within_horizon"]
g = merged["drone_id"]


In [ ]:
categorical_columns = ["model", "motor_type", "firmware_version", "operator_region"]
numerical_columns = X.select_dtypes(include="number").columns
text_column = "maintenance_note"
aux_column = "failure_mode"


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_columns),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_columns),
        ("text", MaintenanceNoteTargetEncoder(smoothing=10), [text_column, aux_column]),
    ],
    remainder="drop",
)


In [32]:
models = [
    LogisticRegression(max_iter=1000),
    RandomForestClassifier(n_estimators=100, random_state=42)
]

In [33]:
transformed = preprocessor.fit_transform(X, y)
transformed_dense = transformed.toarray() if hasattr(transformed, "toarray") else transformed
feature_names = preprocessor.get_feature_names_out()
transformed_df = pd.DataFrame(transformed_dense, columns=feature_names)

In [34]:
for model in models:
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model),
    ])

    scores = cross_val_score(
        pipeline,
        X,
        y,
        groups=g,
        cv=GroupKFold(n_splits=5),
        scoring="average_precision",
    )

    print(f"Model: {model}, Mean: {scores.mean()}, Std: {scores.std()}")


ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "/home/anubi/drones/.venv/lib/python3.11/site-packages/sklearn/model_selection/_validation.py", line 856, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/anubi/drones/.venv/lib/python3.11/site-packages/sklearn/base.py", line 1403, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/anubi/drones/.venv/lib/python3.11/site-packages/sklearn/pipeline.py", line 649, in fit
    self._final_estimator.fit(Xt, y, **last_step_params["fit"])
  File "/home/anubi/drones/.venv/lib/python3.11/site-packages/sklearn/base.py", line 1403, in wrapper